# Music Annotation Project — Notebook 33
## V8.2d Distance-Two Pixel-Ownership Reviewer

**Runtime:** CPU is sufficient.

This development-only Gradio reviewer asks: **who owns the exact dark observed pixel under the purple crosshair?** It merges Notebook 32's high-`p_print` and morphology-ranked distance-two queues, de-duplicates page/y/x, and creates a page-diversified core 100 plus optional extension 100.

### Ownership classes
- **Printed structure:** corresponding genuine print exists at the registered location or under a credible small residual offset.
- **Annotation:** no credible corresponding printed stroke exists there; the observed pixel follows added annotation.
- **Registration ambiguity:** a plausible small residual observed-to-clean displacement changes the ownership answer. Dark observed plus white clean alone is not enough.
- **Mixed / indeterminate ownership:** alignment appears adequate, but overlap/shared edge prevents reliable ownership assignment.

### Derived-truth colours
- **Green:** current deterministic PRINT region.
- **Orange/yellow:** current deterministic UNCERTAIN boundary.
- **Red:** current deterministic ANNOTATION region.

These colours are rule outputs, not confidence or proof. Distance-two candidates normally put the crosshair in red by construction.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "gradio>=5.0" pandas numpy opencv-python-headless pillow openpyxl pyarrow

## Configuration, source resolution and queue construction

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from functools import lru_cache
import json, hashlib, os, platform, sys, threading
import cv2, gradio as gr, numpy as np, pandas as pd
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V82C_ROOT=ROOT/'observed_print_classifier_v8_2c_label_sensitivity'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_2d_ownership_review'
SOURCE_V82C_RUN_OVERRIDE=None
REVIEW_RUN_OVERRIDE=None  # Existing Notebook 33 ownership-review run to resume.
CORE_TARGET=100; EXTENSION_TARGET=100; TOTAL_TARGET=200; CROP_RADIUS=55; SCALE=3
PUBLIC_LINK=True; REVIEWER_NAME='David Bolton'; SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
OWNERS=['Printed structure','Annotation','Registration ambiguity','Mixed / indeterminate ownership']
STRUCTURES=['Not specified','Staff line','Hairpin','Stem','Slur','Text glyph','Other printed structure']
if REVIEW_RUN_OVERRIDE is None:
    OUT=OUTPUT_ROOT/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_2d_ownership_review'); RESUMING=False
else: OUT=Path(REVIEW_RUN_OVERRIDE); RESUMING=True
TABLES=OUT/'tables'; AUDIT=OUT/'audit'
for p in [OUT,TABLES,AUDIT]: p.mkdir(parents=True,exist_ok=RESUMING)
REVIEW_CSV=TABLES/'ownership_review.csv'; REVIEW_XLSX=TABLES/'ownership_review.xlsx'
SUMMARY_CSV=TABLES/'ownership_review_summary.csv'; SUMMARY_XLSX=TABLES/'ownership_review_summary.xlsx'
SAMPLING_CSV=TABLES/'ownership_review_sampling_manifest.csv'; LOCK=threading.RLock()

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def imap(folder):
    return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED}
req=['experiment_identity.json','tables/distance2_annotation_high_pprint.csv','tables/distance2_annotation_review_queue.csv']
def completed(root): return sorted([p for p in Path(root).iterdir() if p.is_dir() and all((p/r).exists() for r in req)],key=lambda p:p.stat().st_mtime) if Path(root).exists() else []
SOURCE=Path(SOURCE_V82C_RUN_OVERRIDE) if SOURCE_V82C_RUN_OVERRIDE else (completed(V82C_ROOT)[-1] if completed(V82C_ROOT) else None)
if SOURCE is None: raise RuntimeError('No complete Notebook 32 run found')
sid=json.loads((SOURCE/'experiment_identity.json').read_text())
if sid.get('version')!='8.2c-whole-population-label-sensitivity-audit': raise RuntimeError(f"Unexpected source {sid.get('version')}")
HP=SOURCE/'tables/distance2_annotation_high_pprint.csv'; MP=SOURCE/'tables/distance2_annotation_review_queue.csv'
high=pd.read_csv(HP); morph=pd.read_csv(MP)
for name,d in [('high',high),('morph',morph)]:
    miss={'page','x','y','p_print'}-set(d.columns)
    if miss: raise RuntimeError(f'{name} missing {miss}')
    d['page']=d.page.astype(str); d['x']=pd.to_numeric(d.x).astype(int); d['y']=pd.to_numeric(d.y).astype(int); d['p_print']=pd.to_numeric(d.p_print).astype(float)
    d['candidate_id']=[hashlib.sha256(f'{p}:{y}:{x}'.encode()).hexdigest()[:20] for p,y,x in zip(d.page,d.y,d.x)]

def diverse(d,score=None):
    q=d.sort_values(score,ascending=False,kind='stable') if score else d.copy()
    groups={p:g.to_dict('records') for p,g in q.groupby('page',sort=False)}; cur={p:0 for p in groups}; out=[]
    while True:
        add=False
        for p in groups:
            if cur[p]<len(groups[p]): out.append(groups[p][cur[p]]); cur[p]+=1; add=True
        if not add: return out
hid=set(high.candidate_id); mid=set(morph.candidate_id)
union=pd.concat([morph,high],ignore_index=True,sort=False).groupby('candidate_id',as_index=False,sort=False).first().set_index('candidate_id',drop=False)
a=diverse(high,'p_print'); b=diverse(morph); seq=[]; seen=set(); i=j=0
while len(seq)<min(TOTAL_TARGET,len(union)) and (i<len(a) or j<len(b)):
    if i<len(a):
        cid=a[i]['candidate_id']; i+=1
        if cid not in seen: seq.append(cid); seen.add(cid)
    if j<len(b) and len(seq)<min(TOTAL_TARGET,len(union)):
        cid=b[j]['candidate_id']; j+=1
        if cid not in seen: seq.append(cid); seen.add(cid)
q=union.loc[seq].copy().reset_index(drop=True); q['queue_order']=np.arange(1,len(q)+1)
q['sampling_stage']=np.where(q.queue_order<=CORE_TARGET,'core_100','extension_100')
q['in_high_probability_queue']=q.candidate_id.isin(hid); q['in_morphology_queue']=q.candidate_id.isin(mid)
q['sampling_origin']=np.select([q.in_high_probability_queue&q.in_morphology_queue,q.in_high_probability_queue,q.in_morphology_queue],['both','high_probability','morphology_ranked'],'unknown')
q['ownership_class']=''; q['structure_type']='Not specified'; q['review_note']=''; q['reviewer']=''; q['reviewed_utc']=''; q['review_revision']=0
q.to_csv(SAMPLING_CSV,index=False); reviews=q.copy()
if RESUMING and REVIEW_CSV.exists():
    old=pd.read_csv(REVIEW_CSV).fillna(''); f=['candidate_id','ownership_class','structure_type','review_note','reviewer','reviewed_utc','review_revision']
    reviews=reviews.drop(columns=f[1:]).merge(old[f],on='candidate_id',how='left',validate='one_to_one')
    for c in f[1:-1]: reviews[c]=reviews[c].fillna('')
    reviews['structure_type']=reviews.structure_type.replace('','Not specified'); reviews['review_revision']=pd.to_numeric(reviews.review_revision,errors='coerce').fillna(0).astype(int)
CLEAN=imap(ROOT/'source/clean'); OBS=imap(ROOT/'aligned')
if set(reviews.page)-set(CLEAN) or set(reviews.page)-set(OBS): raise RuntimeError('Missing clean or observed page')
print('Source',SOURCE,'review rows',len(reviews),'output',OUT)

## Rendering, persistence and reviewer

In [ ]:
TRUTH_BGR={0:(0,0,220),1:(0,200,0),2:(0,190,255)}
def inkmap(clean): return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def tmaps(clean):
    ink=inkmap(clean); k=np.ones((3,3),np.uint8); near=cv2.dilate(ink.astype(np.uint8),k)>0; er=cv2.erode(ink.astype(np.uint8),k)>0
    return ink,near,er,np.logical_xor(near,er)
def crop(a,y,x,r=CROP_RADIUS,pad=255):
    h,w=a.shape[:2]; y0,y1=y-r,y+r+1; x0,x1=x-r,x+r+1; t=max(0,-y0); l=max(0,-x0); b=max(0,y1-h); rr=max(0,x1-w)
    return cv2.copyMakeBorder(a[max(0,y0):min(h,y1),max(0,x0):min(w,x1)],t,b,l,rr,cv2.BORDER_CONSTANT,value=pad)
@lru_cache(maxsize=12)
def ctx(page):
    o=cv2.imread(str(OBS[page]),0); c=cv2.imread(str(CLEAN[page]),0)
    if o is None or c is None or o.shape!=c.shape: raise RuntimeError(f'{page} image failure')
    return o,c,*tmaps(c)
def cross(a,size=19,width=2):
    z=a.copy(); cv2.drawMarker(z,(z.shape[1]//2,z.shape[0]//2),(255,0,255),cv2.MARKER_CROSS,size,width,cv2.LINE_AA); return z
def regpanel(oc,ic,dx,dy):
    z=cv2.cvtColor(oc,cv2.COLOR_GRAY2BGR); edge=cv2.morphologyEx(ic.astype(np.uint8),cv2.MORPH_GRADIENT,np.ones((3,3),np.uint8))>0; z[edge]=(0,255,0)
    c=(CROP_RADIUS,CROP_RADIUS); n=(c[0]+int(dx),c[1]+int(dy)); cv2.drawMarker(z,c,(255,0,255),cv2.MARKER_CROSS,15,1)
    if 0<=n[0]<z.shape[1] and 0<=n[1]<z.shape[0]: cv2.circle(z,n,4,(255,255,0),1); cv2.line(z,c,n,(255,255,0),1)
    return z
def truthpanel(near,bound,y,x):
    a=np.zeros(near.shape,np.uint8); a[near&~bound]=1; a[bound]=2; p=crop(a,y,x,pad=0); z=np.full((*p.shape,3),235,np.uint8)
    for k,v in TRUTH_BGR.items(): z[p==k]=v
    return cross(z,15,1)
def picture(r):
    page=str(r.page); x=int(r.x); y=int(r.y); o,c,ink,near,er,bound=ctx(page); oc=crop(o,y,x); cc=crop(c,y,x); ic=crop(ink.astype(np.uint8),y,x,pad=0)>0
    nx=int(r.get('nearest_clean_ink_x',x)) if pd.notna(r.get('nearest_clean_ink_x',np.nan)) else x; ny=int(r.get('nearest_clean_ink_y',y)) if pd.notna(r.get('nearest_clean_ink_y',np.nan)) else y
    ps=[cv2.cvtColor(oc,cv2.COLOR_GRAY2BGR),cv2.cvtColor(cc,cv2.COLOR_GRAY2BGR),regpanel(oc,ic,nx-x,ny-y),truthpanel(near,bound,y,x)]
    ps=[cv2.resize(p,(p.shape[1]*SCALE,p.shape[0]*SCALE),interpolation=cv2.INTER_NEAREST) for p in ps]; ps[0]=cross(ps[0]); ps[1]=cross(ps[1])
    ph,pw=ps[0].shape[:2]; header=100; sh=np.full((ph+header,pw*4,3),255,np.uint8)
    for i,(p,t) in enumerate(zip(ps,['Observed','Clean','Observed + clean contour','Derived truth'])): sh[header:,i*pw:(i+1)*pw]=p; cv2.putText(sh,t,(i*pw+6,21),0,.46,(20,20,20),1,cv2.LINE_AA)
    lines=[f'{page} x={x} y={y} p_print={float(r.p_print):.6f} origin={r.sampling_origin}',f'stage={r.sampling_stage} distance=2 PURPLE=candidate CYAN=nearest clean ink','Truth: GREEN=PRINT ORANGE=UNCERTAIN RED=ANNOTATION rule (not confidence)']
    for k,t in enumerate(lines): cv2.putText(sh,t,(6,45+22*k),0,.42,(20,20,20),1,cv2.LINE_AA)
    return cv2.cvtColor(sh,cv2.COLOR_BGR2RGB)
def summary():
    d=reviews[reviews.ownership_class.astype(str).str.len()>0]
    if d.empty: return pd.DataFrame(columns=['ownership_class','structure_type','reviews','fraction'])
    s=d.groupby(['ownership_class','structure_type'],as_index=False).agg(reviews=('candidate_id','size'),mean_p_print=('p_print','mean'),median_p_print=('p_print','median'),core=('sampling_stage',lambda x:int((x=='core_100').sum())),extension=('sampling_stage',lambda x:int((x=='extension_100').sum())))
    s['fraction']=s.reviews/len(d); return s
def persist():
    for dest,data in [(REVIEW_CSV,reviews),(SUMMARY_CSV,summary())]:
        t=dest.with_suffix(dest.suffix+'.tmp'); data.to_csv(t,index=False); os.replace(t,dest)
    t=REVIEW_XLSX.with_name(REVIEW_XLSX.stem+'.tmp.xlsx'); reviews.to_excel(t,index=False,engine='openpyxl'); os.replace(t,REVIEW_XLSX)
    t=SUMMARY_XLSX.with_name(SUMMARY_XLSX.stem+'.tmp.xlsx')
    with pd.ExcelWriter(t,engine='openpyxl') as w: summary().to_excel(w,sheet_name='ownership_summary',index=False); reviews.to_excel(w,sheet_name='all_reviews',index=False)
    os.replace(t,SUMMARY_XLSX)
def progress():
    d=reviews.ownership_class.astype(str).str.len()>0; c=reviews.sampling_stage.eq('core_100'); return f'**Reviewed:** {int(d.sum())}/{len(reviews)} | **Core:** {int((d&c).sum())}/{int(c.sum())} | **Extension:** {int((d&~c).sum())}/{int((~c).sum())}'
persist()
FILTERS=['All candidates','Unreviewed only','Reviewed only','Core 100','Extension 100','High-probability queue','Morphology-ranked queue','Present in both queues']
def idxs(f):
    m=pd.Series(True,index=reviews.index)
    if f=='Unreviewed only': m&=reviews.ownership_class.astype(str).str.len().eq(0)
    elif f=='Reviewed only': m&=reviews.ownership_class.astype(str).str.len().gt(0)
    elif f=='Core 100': m&=reviews.sampling_stage.eq('core_100')
    elif f=='Extension 100': m&=reviews.sampling_stage.eq('extension_100')
    elif f=='High-probability queue': m&=reviews.in_high_probability_queue.astype(bool)
    elif f=='Morphology-ranked queue': m&=reviews.in_morphology_queue.astype(bool)
    elif f=='Present in both queues': m&=reviews.in_high_probability_queue.astype(bool)&reviews.in_morphology_queue.astype(bool)
    return reviews.index[m].tolist()
def render(f,pos):
    ids=idxs(f)
    if not ids: return np.full((420,1100,3),245,np.uint8),'No matching records',progress(),'','Not specified','',0,''
    pos=max(0,min(int(pos),len(ids)-1)); r=reviews.loc[ids[pos]]
    meta='\n'.join([f'- **Record:** {pos+1}/{len(ids)}',f'- **Page:** {r.page}',f'- **Coordinate:** ({int(r.x)}, {int(r.y)})',f'- **p_print:** {float(r.p_print):.8f}',f'- **Origin:** {r.sampling_origin}',f"- **Current review:** {r.ownership_class or 'unreviewed'}"])
    return picture(r),meta,progress(),str(r.ownership_class),str(r.structure_type),str(r.review_note),pos,str(r.candidate_id)
def move(f,p,d):
    ids=idxs(f); return render(f,0 if not ids else (int(p)+d)%len(ids))
def save(cid,owner,stype,note,f,p,advance):
    if not cid or owner not in OWNERS or stype not in STRUCTURES: return render(f,p)
    with LOCK:
        hit=reviews.index[reviews.candidate_id.eq(cid)].tolist()
        if len(hit)!=1: raise RuntimeError('Candidate resolution failure')
        i=hit[0]; reviews.at[i,'ownership_class']=owner; reviews.at[i,'structure_type']=stype if owner=='Printed structure' else 'Not specified'; reviews.at[i,'review_note']=str(note or ''); reviews.at[i,'reviewer']=REVIEWER_NAME; reviews.at[i,'reviewed_utc']=datetime.now(timezone.utc).isoformat(); reviews.at[i,'review_revision']=int(reviews.at[i,'review_revision'])+1; persist()
    if not advance:
        return render(f,p)
    # Under 'Unreviewed only' the saved row disappears from the filtered list,
    # so the next unseen item moves into the same position. Advancing by +1 would skip it.
    return render(f,p) if f=='Unreviewed only' else move(f,p,1)
def clear(cid,f,p):
    with LOCK:
        hit=reviews.index[reviews.candidate_id.eq(cid)].tolist()
        if len(hit)==1:
            i=hit[0]
            for c in ['ownership_class','review_note','reviewer','reviewed_utc']: reviews.at[i,c]=''
            reviews.at[i,'structure_type']='Not specified'; reviews.at[i,'review_revision']=int(reviews.at[i,'review_revision'])+1; persist()
    return render(f,p)
def export(): persist(); return [str(REVIEW_CSV),str(REVIEW_XLSX),str(SUMMARY_CSV),str(SUMMARY_XLSX),str(SAMPLING_CSV)]

## Launch

In [ ]:
help_text=(
"Judge the owner of the **exact purple-crosshaired observed dark pixel**, not merely the nearby object.\n\n"
"### Ownership guide\n"
"- **Printed structure:** genuine print corresponds at the same location or under a credible small residual offset.\n"
"- **Annotation:** no credible matching printed stroke exists there; the pixel follows added annotation.\n"
"- **Registration ambiguity:** a plausible small observed-to-clean displacement changes the answer. Dark observed plus white clean alone is not enough.\n"
"- **Mixed / indeterminate ownership:** registration appears adequate, but overlap/shared edge prevents reliable assignment.\n\n"
"**Truth colours:** green = PRINT rule; orange = UNCERTAIN boundary; red = ANNOTATION rule. **Red is not confidence or proof.** Distance-two cases are normally red by construction."
)
with gr.Blocks(title='Notebook 33 Ownership Reviewer') as demo:
    gr.Markdown('# Notebook 33 — Distance-Two Pixel-Ownership Reviewer'); gr.Markdown(help_text); prog=gr.Markdown(progress())
    with gr.Row(): filt=gr.Dropdown(FILTERS,value='Unreviewed only',label='Filter'); rec=gr.Number(value=1,precision=0,label='Jump'); jump=gr.Button('Jump'); refresh=gr.Button('Refresh')
    image=gr.Image(label='Purple crosshair = exact candidate',interactive=False,height=650)
    with gr.Row():
        meta=gr.Markdown()
        with gr.Column(): owner=gr.Radio(OWNERS,label='Pixel ownership'); stype=gr.Dropdown(STRUCTURES,value='Not specified',label='Optional printed structure type'); note=gr.Textbox(label='Optional note',lines=3); advance=gr.Checkbox(value=True,label='Advance after saving')
    with gr.Row(): prev=gr.Button('Previous'); save_next=gr.Button('Save & Next',variant='primary'); save_here=gr.Button('Save current'); clear_btn=gr.Button('Clear'); nxt=gr.Button('Next')
    pos=gr.State(0); cid=gr.State(''); export_btn=gr.Button('Export files'); files=gr.Files()
    outs=[image,meta,prog,owner,stype,note,pos,cid]
    demo.load(lambda:render('Unreviewed only',0),outputs=outs); filt.change(lambda f:render(f,0),inputs=filt,outputs=outs); refresh.click(render,inputs=[filt,pos],outputs=outs)
    jump.click(lambda f,n:render(f,max(0,int(n or 1)-1)),inputs=[filt,rec],outputs=outs); prev.click(lambda f,p:move(f,p,-1),inputs=[filt,pos],outputs=outs); nxt.click(lambda f,p:move(f,p,1),inputs=[filt,pos],outputs=outs)
    save_next.click(save,inputs=[cid,owner,stype,note,filt,pos,advance],outputs=outs); save_here.click(lambda c,o,s,n,f,p:save(c,o,s,n,f,p,False),inputs=[cid,owner,stype,note,filt,pos],outputs=outs); clear_btn.click(clear,inputs=[cid,filt,pos],outputs=outs); export_btn.click(export,outputs=files)
record={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.2d-distance-two-ownership-review','source_notebook32_run':str(SOURCE),'source_identity_sha256':sha(SOURCE/'experiment_identity.json'),'ownership_classes':OWNERS,'structure_types':STRUCTURES,'truth_colours':{'green':'PRINT rule','orange':'UNCERTAIN boundary','red':'ANNOTATION rule; not confidence'},'development_only':True,'retrained':False,'review_targets':{'core':CORE_TARGET,'extension':EXTENSION_TARGET,'actual':len(reviews)}}
(OUT/'experiment_identity.json').write_text(json.dumps(record,indent=2)); print(json.dumps(record,indent=2))
demo.queue(default_concurrency_limit=1).launch(share=PUBLIC_LINK,debug=True)

## Outputs to bring back
- `tables/ownership_review.csv`
- `tables/ownership_review_summary.csv`
- `tables/ownership_review_sampling_manifest.csv`
- `experiment_identity.json`